In [ ]:
%reset -f
%load_ext autoreload
%autoreload 2

import mechanics
from mechanics import *
from mechanics.lagrange import euler_lagrange_equation
import mechanics.space as space

t, = base_spaces('t')
def dot(f): return diff(f, t)

i, j = indices('i j')

x, y = q = variables('x y', i, t)
dq = tuple(dot(q_i) for q_i in q)
ddq = tuple(dot(dq_i) for dq_i in dq)

mu, N = constants(r'\mu N')
m, = constants('m', i)

U = Sum(m * mu / (sqrt((x[i] - x[j])**2 + (y[i] - y[j])**2)), 
        (i, 1, N), (j, i+1, N))
T = Sum(m / 2 * (dot(x)**2 + dot(y)**2), (i, 1, N))
E = T + U
L = T - U
show('L =', L)

EL = euler_lagrange_equation(L, q)
show_equations(EL)

# F = solve(EL, ddq)
# show_equations(F)


In [ ]:
q

In [ ]:
# from mechanics.integrator.runge_kutta import rk4_explicit
from mechanics.integrator.euler import euler_explicit

h, T = constants('h T')
i, = indices('i')
X, euler_step, d = euler_explicit(F, h, i)
show_equations(euler_step)

theta_1, theta_2 = d(q)
v_theta_1, v_theta_2 = d(dq)

In [ ]:
with Solver() as solver:
    solver.constants(m, l, index=(n, 1, 2))
    solver.constants(g, h, T)
    solver.variables(*X, indices=((n, 1, 2), (i, 0, T/h)))
    solver.functions('x_1', 'y_1', 'x_2', 'y_2', 'E', index=(i, 0, T/h))
    solver.inputs(*(x[:,0] for x in X))

    with solver.steps(i, 0, T/h) as step:
        step.solve_explicit(euler_step)
        step.calculate({
            'x_1': d(x_1), 'y_1': d(y_1), 'x_2': d(x_2), 'y_2': d(y_2), 'E': d(E)
            }, i)

In [ ]:
_ = solver.run({
    m: [1.0, 1.0], l: [1.0, 1.0], g: 1.0,
    h: 0.1, T: 1.0,
    theta_1[0]: 1.0, theta_2[0]: 1.0,
    v_theta_1[0]: 0.0, v_theta_2[0]: 0.0,
})


In [ ]:
_